In [ ]:
! pip install torch

In [ ]:
! pip install torchvision

In [ ]:
! pip install unsloth

In [ ]:
! pip install albumentations

Managing memory more flexibly to avoid fragmentation

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

 Loading images even if they're slightly corrupted instead of crashing

In [2]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

### Installation

Detecting the PyTorch version and sets the matching xformers version to install

In [3]:
import re
import torch
v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
print(f"PyTorch: {v} → installing {xformers}")

PyTorch: 2.10 → installing xformers==0.0.29.post3


Installing the needed dependencies

In [ ]:
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

### Unsloth

Let's prepare the OCR model to our local first

Download DeepSeek-OCR files from haggingface

In [4]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR", local_dir = "deepseek_ocr")

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

'/home/jovyan/deepseek_ocr'

Loads the DeepSeek-OCR model with Unsloth's optimizations

Loading all image/transcription (RIMES + collected) for train/test

In [4]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.   
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
    max_seq_length = 4096,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 NVL MIG 1g.24gb. Num GPUs = 1. Max memory: 21.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import os, random
random.seed(42)

collected_2016 = "data/Coll/2016"
collected_2021 = "data/Coll/2021"
rimes_pages    = "data/Rimes"

images_path_2016  = os.path.join(collected_2016, "images")
texts_path_2016   = os.path.join(collected_2016, "transcriptions")
images_path_2021  = os.path.join(collected_2021, "images")
texts_path_2021   = os.path.join(collected_2021, "transcriptions")
images_path_rimes = os.path.join(rimes_pages, "images")
texts_path_rimes  = os.path.join(rimes_pages, "transcriptions")

images_2016  = sorted([os.path.join(images_path_2016,  f) for f in os.listdir(images_path_2016)])
texts_2016   = sorted([os.path.join(texts_path_2016,   f) for f in os.listdir(texts_path_2016)])
images_2021  = sorted([os.path.join(images_path_2021,  f) for f in os.listdir(images_path_2021)])
texts_2021   = sorted([os.path.join(texts_path_2021,   f) for f in os.listdir(texts_path_2021)])
images_rimes = sorted([os.path.join(images_path_rimes, f) for f in os.listdir(images_path_rimes)])
texts_rimes  = sorted([os.path.join(texts_path_rimes,  f) for f in os.listdir(texts_path_rimes)])

# --- RIMES: 200 test, 600 train ---
all_rimes_idx  = list(range(len(images_rimes)))
idx_rimes_test = random.sample(all_rimes_idx, 200)
idx_rimes_train = [i for i in all_rimes_idx if i not in idx_rimes_test]

# --- Collected: 25 test, rest train ---
idx_2016_test = random.sample(range(len(images_2016)), 25)
idx_2021_test = random.sample(range(len(images_2021)), 25)

# --- Test sets ---
test_images = [images_rimes[i] for i in idx_rimes_test] + \
              [images_2016[i]  for i in idx_2016_test]  + \
              [images_2021[i]  for i in idx_2021_test]
test_texts  = [texts_rimes[i]  for i in idx_rimes_test] + \
              [texts_2016[i]   for i in idx_2016_test]  + \
              [texts_2021[i]   for i in idx_2021_test]

# --- Train sets ---
train_images_rimes = [images_rimes[i] for i in idx_rimes_train]
train_texts_rimes  = [texts_rimes[i]  for i in idx_rimes_train]
train_images_2016  = [p for i,p in enumerate(images_2016) if i not in idx_2016_test]
train_texts_2016   = [p for i,p in enumerate(texts_2016)  if i not in idx_2016_test]
train_images_2021  = [p for i,p in enumerate(images_2021) if i not in idx_2021_test]
train_texts_2021   = [p for i,p in enumerate(texts_2021)  if i not in idx_2021_test]

# --- Oversample collected 3x ---
images = train_images_rimes + (train_images_2016 + train_images_2021)
texts  = train_texts_rimes  + (train_texts_2016  + train_texts_2021)

print(f"Train : {len(images)} ({len(train_images_rimes)} RIMES + {len(train_images_2016)+len(train_images_2021)} collected)")
print(f"Test  : {len(test_images)} ({len(idx_rimes_test)} RIMES + {len(idx_2016_test)+len(idx_2021_test)} collected)")

Train : 736 (608 RIMES + 128 collected)
Test  : 250 (200 RIMES + 50 collected)


Copy the transcriptions to the working space

In [6]:
import os, shutil

transcriptions_dir = "data_baseline/all/transcriptions"
os.makedirs(transcriptions_dir, exist_ok=True)

seen_paths = set()
unique_txt_paths = []
for path in texts + test_texts:
    if path not in seen_paths:
        seen_paths.add(path)
        unique_txt_paths.append(path)

for txt_path in unique_txt_paths:
    shutil.copy(txt_path, os.path.join(transcriptions_dir, os.path.basename(txt_path)))

print(f"Copied {len(unique_txt_paths)} transcription files → {transcriptions_dir}")

def to_working_path(p):
    return os.path.join(transcriptions_dir, os.path.basename(p))

texts      = [to_working_path(p) for p in texts]
test_texts = [to_working_path(p) for p in test_texts]

print("Text paths updated.")

Copied 986 transcription files → data_aug/all/transcriptions
Text paths updated.


Convert images to JPG and copy them to the working space

In [7]:
from PIL import Image, ImageFile
from pathlib import Path
ImageFile.LOAD_TRUNCATED_IMAGES = True

output_folder = "data_baseline/all/images"
os.makedirs(output_folder, exist_ok=True)
corrupted = []

def convert_to_jpg(src_path, dst_path):
    try:
        with Image.open(src_path) as im:
            im.convert('RGB').save(dst_path, quality=95)
    except Exception as e:
        corrupted.append((src_path, str(e)))

all_img_paths = list(set(images + test_images))   # ← removed eval_images
for img_path in all_img_paths:
    out = Path(output_folder) / f"{Path(img_path).stem}.jpg"
    convert_to_jpg(img_path, out)

images      = [str(Path(output_folder) / f"{Path(p).stem}.jpg") for p in images]
test_images = [str(Path(output_folder) / f"{Path(p).stem}.jpg") for p in test_images]  # ← removed eval_images

print(f"Converted {len(all_img_paths)} images")
print(f"Corrupted ({len(corrupted)}):")
for path, error in corrupted:
    print(f"  {path} → {error}")

print(f"\nTrain  : {len(images)}")
print(f"Test   : {len(test_images)}")

missing = [(p,'image') for p in images      if not os.path.exists(p)] + \
          [(p,'text')  for p in texts        if not os.path.exists(p)] + \
          [(p,'image') for p in test_images  if not os.path.exists(p)] + \
          [(p,'text')  for p in test_texts   if not os.path.exists(p)]   # ← removed eval lines
print(f"Missing files: {len(missing)}")

Converted 986 images
Corrupted (0):

Train  : 736
Test   : 250
Missing files: 0


Testing some prompts

In [12]:
image_file = images[200]
image_file

'data/all/images/page_339.jpg'

### Test


In [ ]:
import numpy as np
token_counts = []
for txt_path in test_texts:
    with open(txt_path, 'r', encoding='utf-8') as f:
        token_counts.append(len(tokenizer.encode(f.read())))
print(f"Mean: {np.mean(token_counts):.0f} | Median: {np.median(token_counts):.0f} | Max: {max(token_counts)}")

In [ ]:
import jiwer, re, torch
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

FastVisionModel.for_inference(model)

def extract_text_only(raw):
    # Discard if hallucination loop detected
    if raw.count('text-decoration') > 10:
        print('css/html')
        return ""
    raw = re.sub(r'<\|ref\|>.*?<\|/ref\|>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'<\|det\|>.*?<\|/det\|>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'<[^>]+>', '', raw)          # remove stray HTML
    raw = re.sub(r'[\w-]+:[\w-]+;', '', raw)   # remove stray CSS
    lines = raw.strip().splitlines()
    seen, deduped = set(), []
    for line in lines:
        line = line.strip()
        if line and line not in seen:
            seen.add(line)
            deduped.append(line)
    return "\n".join(deduped)

# Build collator ONCE — not inside the loop
collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer, model=model,
    image_size=512, base_size=640,
    crop_mode= True, 
    train_on_responses_only=True,
)

def run_inference(img_path):
    image = Image.open(img_path).convert("RGB")
    sample = {"messages": [
        {"role": "<|User|>", "content": "<image>\n<|grounding|>Free OCR French.", "images": [image]},
        {"role": "<|Assistant|>", "content": ""},
    ]}
    processed   = collator.process_single_sample(sample["messages"])
    input_ids   = processed["input_ids"].unsqueeze(0).to(model.device)
    images_seq  = processed["images_seq_mask"].unsqueeze(0).to(model.device)
    images_ori  = processed["images_ori"].unsqueeze(0).to(model.device)
    images_crop = processed["images_crop"].unsqueeze(0).to(model.device)
    images_spat = processed["images_spatial_crop"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            images=[(images_crop.squeeze(0), images_ori.squeeze(0))],
            images_seq_mask=images_seq,
            images_spatial_crop=images_spat,
            max_new_tokens= 1024,   # hard cap — prevents infinite loops ###############
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)

def run_inference_batch(img_paths, max_new_tokens=512):
    """Run inference on a list of images. Returns list of raw predictions."""
    preds = []
    for idx, img_path in enumerate(img_paths):
        try:
            pred = run_inference(img_path)
        except Exception as e:
            print(f'  Inference error on {img_path}: {e}')
            pred = ''
        preds.append(pred)
        if (idx + 1) % 25 == 0:
            print(f'  [{idx+1}/{len(img_paths)}] done...')
    return preds

In [ ]:
import jiwer, os, pandas as pd

def evaluate_detailed(img_list, txt_list, tag=''):
    """Run eval with split WER/CER reporting. Uses batched inference."""

    # --- Run all predictions at once ---
    print(f'Running inference on {len(img_list)} images...')
    raw_preds = run_inference_batch(img_list)

    # --- Score them ---
    records = []
    skipped_ref, skipped_hyp, errors = 0, 0, 0
    os.makedirs('test_results', exist_ok=True)

    for img_path, txt_path, raw_pred in zip(img_list, txt_list, raw_preds):
        try:
            with open(txt_path, 'r', encoding='utf-8') as f:
                reference = extract_text_only(f.read())
            hypothesis = extract_text_only(raw_pred)

            pred_path = os.path.join('test_results', os.path.basename(txt_path))
            with open(pred_path, 'w', encoding='utf-8') as f:
                f.write(hypothesis)

            if not reference.strip():
                skipped_ref += 1; continue
            if not hypothesis.strip():
                skipped_hyp += 1; continue

            wer = jiwer.wer(reference, hypothesis)
            cer = jiwer.cer(reference, hypothesis)
            exact = 1.0 if reference.strip() == hypothesis.strip() else 0.0
            is_rimes = os.path.basename(img_path).startswith('page_')
            source = 'RIMES' if is_rimes else 'Collected'
            fname = os.path.basename(img_path)
            records.append({'source': source, 'file': fname,
                            'wer': wer, 'cer': cer, 'exact': exact})

        except Exception as e:
            print(f'Error scoring {img_path}: {e}')
            errors += 1

    print(f'Skipped — empty ref: {skipped_ref} | empty hyp: {skipped_hyp} | errors: {errors}')

    if not records:
        print('No valid results.'); return None

    df = pd.DataFrame(records)

    # --- Build summary table ---
    rows = []
    for src in ['RIMES', 'Collected', 'Overall']:
        sub = df if src == 'Overall' else df[df['source'] == src]
        if len(sub) == 0: continue
        ok  = sub[sub['wer'] < 1.0]
        bad = sub[sub['wer'] >= 1.0]
        best_row  = sub.loc[sub['wer'].idxmin()]
        worst_row = sub.loc[sub['wer'].idxmax()]
        rows.append({
            'Source': src,
            'N': len(sub),
            'Exact Match %': f"{sub['exact'].mean()*100:.1f}",
            'Readable (WER<100%)': len(ok),
            'Mean WER (readable)':  f"{ok['wer'].mean():.3f}" if len(ok) else '-',
            'Mean CER (readable)':  f"{ok['cer'].mean():.3f}" if len(ok) else '-',
            'Problematic (WER>=100%)': len(bad),
            'Max WER (problematic)':   f"{bad['wer'].max():.2f}" if len(bad) else '-',
            'Best (file | WER)':  f"{best_row['file']} | {best_row['wer']:.3f}",
            'Worst (file | WER)': f"{worst_row['file']} | {worst_row['wer']:.3f}",
        })

    summary = pd.DataFrame(rows).set_index('Source')
    print(f'\n=== {tag} Results ===')
    print(summary.to_string())
    return df

Evaluation loop

In [ ]:
FastVisionModel.for_inference(model)
baseline_results = evaluate_detailed(test_images, test_texts, tag='BASELINE')